# 6.8 — Adaptive Optimizers

Adaptive optimizers change the size of each parameter's step using the history of its own gradients. In this lesson, you will build the ideas from raw gradient descent to AdaGrad, RMSProp, Adam, and AdamW using only NumPy, while keeping an eye on the scale, bias, and numerical stability that make deep-learning training work in practice.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build adaptive optimizers one idea at a time. Run each cell in order and read the printed intermediate values — every moving average, denominator, and parameter update is shown so the optimizer is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, elementwise optimizer arithmetic, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic gradient sequences.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

### 1. From a raw signal to one vanilla gradient step

Optimizers act after a network has converted inputs into a loss and a gradient. To keep the arithmetic visible, start with the lesson's two-input scratch pass: an affine score, a ReLU gate, and then one scalar gradient step. This is not yet adaptive; it is the baseline every adaptive method modifies.

In [ ]:
x_w = np.array([1.5, -0.5])  # two inputs entering a tiny neuron.
w_w = np.array([1.6, -0.7])  # two weights for the affine signal.
b_w = 0.5  # bias from the lesson block.
z_w = float(w_w @ x_w + b_w)  # affine signal = 1.6*1.5 + (-0.7)*(-0.5) + 0.5.
a_w = max(0.0, z_w)  # ReLU gate keeps positive signals and clips negative ones.

print("affine signal:", round(z_w, 3))
print("gated signal:", round(a_w, 3))

assert round(z_w, 3) == 3.25 and round(a_w, 3) == 3.25

▶ What you'll see: the affine signal is `3.25`, and the ReLU passes it through unchanged because it is positive.

In [ ]:
theta_w = 2.0  # one scalar parameter before the update.
eta_w = 0.08  # learning rate from the lesson block.
g_w = 1.35  # local gradient from the lesson block.
theta_next_w = theta_w - eta_w * g_w  # vanilla gradient descent step.

print("step size:", round(eta_w * g_w, 3))
print("theta after vanilla GD:", round(theta_next_w, 3))

assert round(theta_next_w, 3) == 1.892

▶ What you'll see: the parameter moves from `2.000` to `1.892`, a small reliable nudge.

In [ ]:
scores_w = np.array([z_w, 0.4])  # compare the lesson score against a baseline score.
exp_w = np.exp(scores_w)  # exponentiate scores before normalizing.
prob_w = exp_w[0] / exp_w.sum()  # two-class softmax probability for the lesson score.

print("exp scores:", np.round(exp_w, 3))
print("softmax probability:", round(float(prob_w), 3))

assert round(float(prob_w), 3) == 0.945

▶ What you'll see: a score of `3.25` becomes a high probability (`0.945`) only after comparison with the baseline.

*Why it's done this way:* gradient descent needs a direction and a scale. The gradient says which way locally increases the loss, so subtracting `ηg` moves downhill; the learning rate is deliberately small because many consistent local moves are safer than one large leap. Adaptive optimizers keep this same subtraction structure but replace the raw scalar step with a coordinatewise, history-aware step.

### 2. Coordinatewise scaling: why one learning rate is not enough

A vector parameter can have coordinates with very different gradient magnitudes. If one coordinate repeatedly receives large gradients and another receives tiny gradients, the same learning rate is too aggressive for one and too timid for the other. Adaptive optimizers fix this by dividing each coordinate by a scale estimate built from its own gradient history.

In [ ]:
theta2_w = np.array([2.0, 2.0])  # two parameters sharing one global learning rate.
g2_w = np.array([1.35, 0.05])  # one large gradient coordinate and one tiny coordinate.
vanilla_step_w = 0.08 * g2_w  # vanilla GD uses the same eta for both coordinates.

print("raw gradients:", g2_w)
print("vanilla steps:", np.round(vanilla_step_w, 4))

▶ What you'll see: the first coordinate moves `27×` more than the second because its gradient is `27×` larger.

In [ ]:
scale2_w = np.sqrt(g2_w ** 2) + 1e-8  # simplest per-coordinate scale from the current gradient magnitude.
adapt_step_w = 0.08 * g2_w / scale2_w  # normalized step uses sign-like equalized magnitudes.

print("per-coordinate scale:", np.round(scale2_w, 4))
print("normalized adaptive steps:", np.round(adapt_step_w, 4))

assert np.allclose(np.round(adapt_step_w, 3), [0.08, 0.08])

▶ What you'll see: after dividing by each coordinate's own magnitude, both coordinates get comparable step sizes.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["large grad", "tiny grad"], vanilla_step_w, width=0.4, label="vanilla", color="gray")
plt.bar(np.arange(2) + 0.4, adapt_step_w, width=0.4, label="scaled", color="seagreen")
plt.xticks(np.arange(2) + 0.2, ["coord 0", "coord 1"])
plt.ylabel("absolute update size")
plt.title("2: coordinatewise scaling changes effective steps")
plt.legend()
plt.show()

▶ What you'll see: vanilla steps mirror gradient size, while scaled steps are balanced by each coordinate's denominator.

*Why it's done this way:* a single learning rate assumes all coordinates live on the same numerical scale. Dividing by a coordinate-specific magnitude estimate is a diagonal preconditioner: it shrinks directions that have historically steep gradients and protects directions with small but persistent gradients from being ignored.

### 3. AdaGrad: remember all squared gradients

AdaGrad is the simplest historical adaptive optimizer. It accumulates squared gradients, $G_t=G_{t-1}+g_t^2$, then updates with $\theta_t=\theta_{t-1}-\eta g_t/(\sqrt{G_t}+\epsilon)$. Coordinates that frequently receive large gradients build large denominators, so their effective learning rate decays quickly.

In [ ]:
grads_ag_w = np.array([[1.2, 0.10], [1.0, 0.10], [0.8, 0.10], [0.6, 0.10]])  # repeated gradients.
G_ag_w = np.zeros(2)  # accumulated squared-gradient memory.
eta_ag_w = 0.4  # intentionally visible learning rate for the toy example.
steps_ag_w = []  # store coordinatewise updates for inspection.
for g_step_w in grads_ag_w:
    G_ag_w += g_step_w ** 2  # AdaGrad never forgets past squared gradients.
    steps_ag_w.append(eta_ag_w * g_step_w / (np.sqrt(G_ag_w) + 1e-8))
steps_ag_w = np.array(steps_ag_w)

print("final accumulated G:", np.round(G_ag_w, 3))
print("AdaGrad steps:\n", np.round(steps_ag_w, 3))

▶ What you'll see: the first coordinate's denominator grows much faster, so its step shrinks from `0.400` to about `0.137`.

In [ ]:
assert np.allclose(np.round(G_ag_w, 2), [3.44, 0.04])
assert round(float(steps_ag_w[-1, 0]), 3) == 0.129
plt.figure(figsize=(5, 3))
plt.plot(steps_ag_w[:, 0], marker="o", label="coord 0: frequent large gradients")
plt.plot(steps_ag_w[:, 1], marker="s", label="coord 1: small gradients")
plt.title("3: AdaGrad step sizes decay with accumulated squares")
plt.xlabel("time step")
plt.ylabel("effective update")
plt.legend()
plt.show()

▶ What you'll see: the large-gradient coordinate decays strongly, while the small-gradient coordinate stays near the learning-rate scale.

*Why it's done this way:* summing $g^2$ makes the denominator a memory of how much curvature/noise each coordinate has experienced. The square removes signs so positive and negative gradients both count as volatility. The tradeoff is that the denominator only grows, which can make AdaGrad's steps become too small late in training.

### 4. RMSProp: forget old squared gradients with an exponential average

RMSProp changes AdaGrad's permanent memory into a leaky memory: $v_t=\beta v_{t-1}+(1-\beta)g_t^2$. Recent squared gradients matter most; old ones fade. This keeps the denominator responsive when the training landscape changes.

In [ ]:
grads_rms_w = np.array([1.2, 1.0, 0.2, 0.2, 0.2])  # large early gradients followed by calmer gradients.
beta_rms_w = 0.9  # exponential-memory coefficient.
v_rms_w = 0.0  # second-moment accumulator.
vs_rms_w = []  # store denominators over time.
steps_rms_w = []  # store RMSProp update magnitudes.
for g_step_w in grads_rms_w:
    v_rms_w = beta_rms_w * v_rms_w + (1 - beta_rms_w) * g_step_w ** 2
    vs_rms_w.append(v_rms_w)
    steps_rms_w.append(0.1 * g_step_w / (np.sqrt(v_rms_w) + 1e-8))

print("RMSProp v history:", np.round(vs_rms_w, 4))
print("RMSProp steps:", np.round(steps_rms_w, 3))

▶ What you'll see: the squared-gradient memory rises on large gradients, then gradually adapts downward when gradients become small.

In [ ]:
v_adagrad_like_w = np.cumsum(grads_rms_w ** 2)  # permanent memory for comparison.
steps_ag_like_w = 0.1 * grads_rms_w / (np.sqrt(v_adagrad_like_w) + 1e-8)

print("AdaGrad-like steps:", np.round(steps_ag_like_w, 3))

assert round(float(vs_rms_w[0]), 3) == 0.144
assert round(float(steps_rms_w[0]), 3) == 0.316

▶ What you'll see: RMSProp's first step is large because its exponential average starts at zero, while AdaGrad's permanent accumulator is already the full first square.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(steps_rms_w, marker="o", label="RMSProp leaky memory")
plt.plot(steps_ag_like_w, marker="s", label="AdaGrad permanent memory")
plt.title("4: leaky vs permanent squared-gradient memory")
plt.xlabel("time step")
plt.ylabel("effective update")
plt.legend()
plt.show()

▶ What you'll see: RMSProp adapts more to recent gradient scale, while AdaGrad keeps shrinking from all past gradients.

*Why it's done this way:* deep-learning gradients are nonstationary: early layers, later layers, and minibatches can change scale over training. An exponential average is a compromise between noise reduction and responsiveness, so RMSProp can keep learning after early large gradients instead of letting old history dominate forever.

### 5. Adam: momentum plus RMSProp, with bias correction

Adam combines a first-moment average $m_t$ (momentum-like direction) with a second-moment average $v_t$ (RMSProp-like scale). Because both start at zero, early averages are biased toward zero; Adam corrects them using $\hat m_t=m_t/(1-\beta_1^t)$ and $\hat v_t=v_t/(1-\beta_2^t)$ before stepping.

In [ ]:
grads_adam_w = np.array([1.35, 1.00, -0.20, -0.10])  # a changing one-dimensional gradient sequence.
b1_w, b2_w = 0.9, 0.999  # standard Adam decay rates.
m_adam_w, v_adam_w = 0.0, 0.0  # first and second moment states.
rows_adam_w = []  # store raw and corrected values.
for t_w, g_step_w in enumerate(grads_adam_w, start=1):
    m_adam_w = b1_w * m_adam_w + (1 - b1_w) * g_step_w
    v_adam_w = b2_w * v_adam_w + (1 - b2_w) * g_step_w ** 2
    mhat_w = m_adam_w / (1 - b1_w ** t_w)
    vhat_w = v_adam_w / (1 - b2_w ** t_w)
    rows_adam_w.append([m_adam_w, mhat_w, v_adam_w, vhat_w])
rows_adam_w = np.array(rows_adam_w)

print("raw m vs corrected m_hat:\n", np.round(rows_adam_w[:, :2], 4))
print("raw v vs corrected v_hat:\n", np.round(rows_adam_w[:, 2:], 4))

▶ What you'll see: on the first step, raw `m=0.135` but corrected `m_hat=1.35`, undoing the zero-start bias.

In [ ]:
eta_adam_w = 0.08  # lesson learning-rate scale.
eps_adam_w = 1e-8  # numerical stabilizer.
steps_adam_w = eta_adam_w * rows_adam_w[:, 1] / (np.sqrt(rows_adam_w[:, 3]) + eps_adam_w)

print("Adam corrected steps:", np.round(steps_adam_w, 4))

assert round(float(steps_adam_w[0]), 3) == 0.08

▶ What you'll see: the first Adam step is exactly the learning-rate scale because `m_hat=g` and `sqrt(v_hat)=|g|` at `t=1`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(grads_adam_w, marker="o", label="raw gradient")
plt.plot(rows_adam_w[:, 1], marker="s", label="bias-corrected m_hat")
plt.title("5: Adam smooths direction with a first moment")
plt.xlabel("time step")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: `m_hat` changes sign more slowly than the raw gradient, which dampens noisy reversals.

*Why it's done this way:* $m_t$ remembers direction so one noisy minibatch does not fully reverse the optimizer, while $v_t$ remembers scale so steep coordinates get smaller effective steps. Bias correction matters mathematically because initializing both memories at zero would otherwise make early denominators and numerators systematically too small.

### 6. AdamW: decouple weight decay from the adaptive gradient step

Classic Adam often mixes L2 regularization into the gradient before the adaptive denominator. AdamW separates the two effects: first take the adaptive gradient step, then shrink the weight directly by $\eta\lambda\theta$. This preserves weight decay as a real parameter-size constraint instead of letting the adaptive denominator distort it coordinate by coordinate.

In [ ]:
theta_aw_w = np.array([2.0, -0.5])  # two parameters with different magnitudes.
g_aw_w = np.array([1.35, 0.05])  # current gradients.
vhat_aw_w = g_aw_w ** 2  # make the first-step Adam denominator explicit.
eta_aw_w = 0.08  # lesson learning-rate scale.
wd_aw_w = 0.1  # weight decay strength.
adaptive_aw_w = eta_aw_w * g_aw_w / (np.sqrt(vhat_aw_w) + 1e-8)  # adaptive gradient component.
decay_aw_w = eta_aw_w * wd_aw_w * theta_aw_w  # decoupled shrinkage component.

print("adaptive component:", np.round(adaptive_aw_w, 3))
print("weight-decay component:", np.round(decay_aw_w, 3))

▶ What you'll see: the adaptive gradient component is sign-scaled, while decay is proportional to parameter size.

In [ ]:
theta_next_aw_w = theta_aw_w - adaptive_aw_w - decay_aw_w  # AdamW-style update.

print("theta before:", theta_aw_w)
print("theta after AdamW-style step:", np.round(theta_next_aw_w, 3))

assert np.allclose(np.round(theta_next_aw_w, 3), [1.904, -0.576])

▶ What you'll see: the positive parameter shrinks further toward zero; the negative parameter is also decayed toward zero before the gradient direction is considered.

In [ ]:
normed_value_w = (3.25 - 1.0) / np.sqrt(0.25 + 0.00001)  # lesson normalization arithmetic.
mem_kb_w = 2 * 128 * 4 / 1024  # two float32 activation vectors of length 128.

print("normalized value:", round(float(normed_value_w), 3))
print("activation memory KB:", round(mem_kb_w, 3))

assert round(float(normed_value_w), 3) == 4.5 and round(mem_kb_w, 3) == 1.0

▶ What you'll see: the same raw signal can become a large normalized value, and even tiny activation blocks have concrete memory cost.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["adaptive step 0", "decay step 0"], [adaptive_aw_w[0], decay_aw_w[0]], color=["teal", "orange"])
plt.title("6: AdamW keeps gradient adaptation and decay separate")
plt.ylabel("update component")
plt.show()

▶ What you'll see: the decay term is smaller than the adaptive gradient term here, but it is a separate force tied to parameter size.

*Why it's done this way:* adaptive denominators are about gradient scale, while weight decay is about model capacity. Coupling them means the amount of regularization depends on gradient history, which is not the intended penalty. AdamW decouples the two so training stability and capacity control can be tuned more independently.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Vanilla signal and gradient step

Adaptive methods still start from a forward signal and a plain gradient-descent subtraction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t1_x = np.array([1.5, -0.5, 1.0])  # -> [1.5, -0.5, 1.0]

print("inputs:", t1_x.tolist())  # -> [1.5, -0.5, 1.0]

t1_w = np.array([1.0, -0.4, 0.8])  # -> [1.0, -0.4, 0.8]

print("weights:", t1_w.tolist())  # -> [1.0, -0.4, 0.8]

t1_b = 0.2  # -> 0.2

print("bias:", t1_b)  # -> 0.2

t1_pieces = t1_x * t1_w  # -> [1.5, 0.2, 0.8]

print("affine pieces:", t1_pieces.tolist())  # -> [1.5, 0.2, 0.8]

t1_z = np.sum(t1_pieces) + t1_b  # -> 2.7

print("affine score:", round(float(t1_z), 3))  # -> 2.7

t1_relu = np.maximum(0.0, t1_z)  # -> 2.7

print("ReLU output:", round(float(t1_relu), 3))  # -> 2.7

t1_scores = np.array([t1_relu, 0.5, -0.5])  # -> [2.7, 0.5, -0.5]

print("scores:", t1_scores.tolist())  # -> [2.7, 0.5, -0.5]

t1_exp = np.exp(t1_scores - np.max(t1_scores))  # -> [1.0, 0.110803, 0.040762]

print("shifted exp:", np.round(t1_exp, 3).tolist())  # -> [1.0, 0.111, 0.041]

t1_probs = t1_exp / np.sum(t1_exp)  # -> [0.868226, 0.096225, 0.035549]

print("softmax probabilities:", np.round(t1_probs, 3).tolist())  # -> [0.868, 0.096, 0.035]

t1_theta = 2.0  # -> 2.0

print("theta before:", t1_theta)  # -> 2.0

t1_grad = 1.25  # -> 1.25

print("gradient:", t1_grad)  # -> 1.25

t1_eta = 0.08  # -> 0.08

print("learning rate:", t1_eta)  # -> 0.08

t1_step = t1_eta * t1_grad  # -> 0.1

print("vanilla step:", round(float(t1_step), 3))  # -> 0.1

t1_theta_next = t1_theta - t1_step  # -> 1.9

print("theta after:", round(float(t1_theta_next), 3))  # -> 1.9

assert round(float(t1_probs[0]), 3) == 0.868
assert round(float(t1_theta_next), 3) == 1.9

plt.figure(figsize=(4.6, 2.8))
plt.bar(["ReLU", "p0", "theta after"], [t1_relu, t1_probs[0], t1_theta_next], color=["teal", "purple", "seagreen"])
plt.ylabel("value")
plt.title("Toy 1 · baseline arithmetic")
plt.show()

▶ What you'll see: the forward signal gives probability `0.868`, and vanilla GD moves `theta` to `1.9`.

### ✍️ Toy 2 · Coordinatewise scaling equalizes steps

Dividing by a coordinate's own gradient magnitude turns uneven raw steps into sign-sized steps.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t2_grad = np.array([1.2, 0.1, -0.4, 0.05, 0.8, -0.2])  # -> [1.2, 0.1, -0.4, 0.05, 0.8, -0.2]

print("gradients:", t2_grad.tolist())  # -> [1.2, 0.1, -0.4, 0.05, 0.8, -0.2]

t2_eta = 0.06  # -> 0.06

print("learning rate:", t2_eta)  # -> 0.06

t2_vanilla = t2_eta * t2_grad  # -> [0.072, 0.006, -0.024, 0.003, 0.048, -0.012]

print("vanilla steps:", np.round(t2_vanilla, 4).tolist())  # -> [0.072, 0.006, -0.024, 0.003, 0.048, -0.012]

t2_scale = np.sqrt(t2_grad ** 2) + 1e-8  # -> [1.2, 0.1, 0.4, 0.05, 0.8, 0.2]

print("coordinate scales:", np.round(t2_scale, 4).tolist())  # -> [1.2, 0.1, 0.4, 0.05, 0.8, 0.2]

t2_adaptive = t2_eta * t2_grad / t2_scale  # -> [0.06, 0.06, -0.06, 0.06, 0.06, -0.06]

print("scaled steps:", np.round(t2_adaptive, 4).tolist())  # -> [0.06, 0.06, -0.06, 0.06, 0.06, -0.06]

assert np.allclose(np.round(np.abs(t2_adaptive), 3), np.full(6, 0.06))
assert round(float(t2_vanilla[0] / t2_vanilla[1]), 1) == 12.0

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.abs(t2_vanilla), marker="o", label="vanilla |step|", color="gray")
plt.plot(np.abs(t2_adaptive), marker="s", label="scaled |step|", color="seagreen")
plt.xlabel("coordinate")
plt.ylabel("absolute update")
plt.title("Toy 2 · coordinate scaling")
plt.legend()
plt.show()

▶ What you'll see: raw updates vary by `12×`, while scaled updates all have magnitude `0.06`.

### ✍️ Toy 3 · AdaGrad accumulates squared gradients forever

AdaGrad's denominator only grows, so repeated gradients make later steps smaller.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t3_grads = np.array([[1.0, 0.1], [0.8, 0.1], [0.6, 0.1], [0.4, 0.1], [0.2, 0.1], [0.1, 0.1]])  # -> six 2D gradients

print("gradients:\n", t3_grads)  # -> [[1.0,0.1], [0.8,0.1], [0.6,0.1], [0.4,0.1], [0.2,0.1], [0.1,0.1]]

t3_eta = 0.3  # -> 0.3

print("learning rate:", t3_eta)  # -> 0.3

t3_G = np.zeros(2)  # -> [0.0, 0.0]

print("initial accumulator:", t3_G.tolist())  # -> [0.0, 0.0]

t3_history = []
t3_steps = []
for t3_grad in t3_grads:
    t3_G = t3_G + t3_grad ** 2
    t3_step = t3_eta * t3_grad / (np.sqrt(t3_G) + 1e-8)
    t3_history.append(t3_G.copy())
    t3_steps.append(t3_step.copy())
t3_history = np.array(t3_history)  # -> final [2.21, 0.06]

print("G history:\n", np.round(t3_history, 3))  # -> [[1.0,0.01], [1.64,0.02], [2.0,0.03], [2.16,0.04], [2.2,0.05], [2.21,0.06]]

t3_steps = np.array(t3_steps)  # -> final [0.02018, 0.122474]

print("AdaGrad steps:\n", np.round(t3_steps, 3))  # -> [[0.3,0.3], [0.187,0.212], [0.127,0.173], [0.082,0.15], [0.04,0.134], [0.02,0.122]]

t3_final_G = t3_history[-1]  # -> [2.21, 0.06]

print("final G:", np.round(t3_final_G, 3).tolist())  # -> [2.21, 0.06]

assert np.allclose(np.round(t3_final_G, 2), [2.21, 0.06])
assert round(float(t3_steps[-1, 0]), 3) == 0.02

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_steps[:, 0], marker="o", label="coord 0", color="purple")
plt.plot(t3_steps[:, 1], marker="s", label="coord 1", color="gray")
plt.xlabel("time")
plt.ylabel("step")
plt.title("Toy 3 · AdaGrad decay")
plt.legend()
plt.show()

▶ What you'll see: the large-gradient coordinate's step shrinks from `0.3` to `0.02`.

### ✍️ Toy 4 · RMSProp uses leaky squared-gradient memory

RMSProp forgets old squared gradients so its denominator can adjust when gradients calm down.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t4_grads = np.array([1.0, 0.8, 0.2, 0.2, 0.2, 0.2])  # -> [1.0, 0.8, 0.2, 0.2, 0.2, 0.2]

print("gradients:", t4_grads.tolist())  # -> [1.0, 0.8, 0.2, 0.2, 0.2, 0.2]

t4_beta = 0.8  # -> 0.8

print("decay beta:", t4_beta)  # -> 0.8

t4_eta = 0.1  # -> 0.1

print("learning rate:", t4_eta)  # -> 0.1

t4_v = 0.0  # -> 0.0

print("initial v:", t4_v)  # -> 0.0

t4_vs = []
t4_steps = []
for t4_grad in t4_grads:
    t4_v = t4_beta * t4_v + (1.0 - t4_beta) * t4_grad ** 2
    t4_step = t4_eta * t4_grad / (np.sqrt(t4_v) + 1e-8)
    t4_vs.append(t4_v)
    t4_steps.append(t4_step)
t4_vs = np.array(t4_vs)  # -> [0.2, 0.288, 0.2384, 0.1987, 0.167, 0.1416]

print("RMSProp v:", np.round(t4_vs, 4).tolist())  # -> [0.2, 0.288, 0.2384, 0.1987, 0.167, 0.1416]

t4_steps = np.array(t4_steps)  # -> [0.223607, 0.149071, 0.040945, 0.044882, 0.048938, 0.053143]

print("RMSProp steps:", np.round(t4_steps, 3).tolist())  # -> [0.224, 0.149, 0.041, 0.045, 0.049, 0.053]

assert round(float(t4_vs[0]), 3) == 0.2
assert round(float(t4_steps[-1]), 3) == 0.053

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_vs, marker="o", label="v", color="orange")
plt.plot(t4_steps, marker="s", label="step", color="teal")
plt.xlabel("time")
plt.title("Toy 4 · RMSProp leaky memory")
plt.legend()
plt.show()

▶ What you'll see: after early large gradients, the RMSProp memory decays and the small-gradient step grows slightly.

### ✍️ Toy 5 · Adam bias-corrects moments

Adam combines first and second moments, then divides out zero-initialization bias.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t5_grads = np.array([1.2, 1.0, 0.5, -0.2, -0.1, 0.3])  # -> [1.2, 1.0, 0.5, -0.2, -0.1, 0.3]

print("gradients:", t5_grads.tolist())  # -> [1.2, 1.0, 0.5, -0.2, -0.1, 0.3]

t5_beta1 = 0.9  # -> 0.9

print("beta1:", t5_beta1)  # -> 0.9

t5_beta2 = 0.999  # -> 0.999

print("beta2:", t5_beta2)  # -> 0.999

t5_eta = 0.08  # -> 0.08

print("learning rate:", t5_eta)  # -> 0.08

t5_m = 0.0  # -> 0.0
t5_v = 0.0  # -> 0.0

print("initial m,v:", t5_m, t5_v)  # -> 0.0 0.0

t5_rows = []
t5_steps = []
for t5_t, t5_grad in enumerate(t5_grads, start=1):
    t5_m = t5_beta1 * t5_m + (1.0 - t5_beta1) * t5_grad
    t5_v = t5_beta2 * t5_v + (1.0 - t5_beta2) * t5_grad ** 2
    t5_mhat = t5_m / (1.0 - t5_beta1 ** t5_t)
    t5_vhat = t5_v / (1.0 - t5_beta2 ** t5_t)
    t5_step = t5_eta * t5_mhat / (np.sqrt(t5_vhat) + 1e-8)
    t5_rows.append([t5_m, t5_mhat, t5_v, t5_vhat])
    t5_steps.append(t5_step)
t5_rows = np.array(t5_rows)  # -> six rows of raw/corrected moments

print("raw m:", np.round(t5_rows[:, 0], 4).tolist())  # -> [0.12, 0.208, 0.2372, 0.1935, 0.1641, 0.1777]
print("corrected m_hat:", np.round(t5_rows[:, 1], 4).tolist())  # -> [1.2, 1.0947, 0.8753, 0.5626, 0.4008, 0.3793]
print("raw v:", np.round(t5_rows[:, 2], 4).tolist())  # -> [0.0014, 0.0024, 0.0027, 0.0027, 0.0027, 0.0028]
print("corrected v_hat:", np.round(t5_rows[:, 3], 4).tolist())  # -> [1.44, 1.2199, 0.8963, 0.6819, 0.5472, 0.4708]

t5_steps = np.array(t5_steps)  # -> [0.08, 0.0793, 0.074, 0.0545, 0.0433, 0.0442]

print("Adam steps:", np.round(t5_steps, 4).tolist())  # -> [0.08, 0.0793, 0.074, 0.0545, 0.0433, 0.0442]

assert round(float(t5_rows[0, 1]), 3) == 1.2
assert round(float(t5_steps[0]), 3) == 0.08

plt.figure(figsize=(4.8, 2.8))
plt.plot(t5_grads, marker="o", label="gradient", color="gray")
plt.plot(t5_rows[:, 1], marker="s", label="m_hat", color="teal")
plt.xlabel("time")
plt.title("Toy 5 · Adam first moment")
plt.legend()
plt.show()

▶ What you'll see: the first corrected moment jumps back to the real gradient `1.2` instead of staying at raw `0.12`.

### ✍️ Toy 6 · AdamW decouples gradient and decay

AdamW subtracts the adaptive gradient step and the weight-decay shrinkage as two separate terms.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t6_theta = np.array([2.0, -1.0, 0.5, -0.5, 1.5, -2.0])  # -> [2.0, -1.0, 0.5, -0.5, 1.5, -2.0]

print("theta before:", t6_theta.tolist())  # -> [2.0, -1.0, 0.5, -0.5, 1.5, -2.0]

t6_grad = np.array([0.8, -0.2, 0.4, -0.1, 0.5, -0.3])  # -> [0.8, -0.2, 0.4, -0.1, 0.5, -0.3]

print("gradient:", t6_grad.tolist())  # -> [0.8, -0.2, 0.4, -0.1, 0.5, -0.3]

t6_vhat = t6_grad ** 2  # -> [0.64, 0.04, 0.16, 0.01, 0.25, 0.09]

print("v_hat:", np.round(t6_vhat, 3).tolist())  # -> [0.64, 0.04, 0.16, 0.01, 0.25, 0.09]

t6_eta = 0.05  # -> 0.05

print("learning rate:", t6_eta)  # -> 0.05

t6_wd = 0.1  # -> 0.1

print("weight decay:", t6_wd)  # -> 0.1

t6_adaptive = t6_eta * t6_grad / (np.sqrt(t6_vhat) + 1e-8)  # -> [0.05, -0.05, 0.05, -0.05, 0.05, -0.05]

print("adaptive step:", np.round(t6_adaptive, 3).tolist())  # -> [0.05, -0.05, 0.05, -0.05, 0.05, -0.05]

t6_decay = t6_eta * t6_wd * t6_theta  # -> [0.01, -0.005, 0.0025, -0.0025, 0.0075, -0.01]

print("decay step:", np.round(t6_decay, 4).tolist())  # -> [0.01, -0.005, 0.0025, -0.0025, 0.0075, -0.01]

t6_theta_next = t6_theta - t6_adaptive - t6_decay  # -> [1.94, -0.945, 0.4475, -0.4475, 1.4425, -1.94]

print("theta after:", np.round(t6_theta_next, 4).tolist())  # -> [1.94, -0.945, 0.4475, -0.4475, 1.4425, -1.94]

assert np.allclose(np.round(t6_theta_next, 4), [1.94, -0.945, 0.4475, -0.4475, 1.4425, -1.94])
assert np.allclose(np.round(np.abs(t6_adaptive), 3), np.full(6, 0.05))

plt.figure(figsize=(4.8, 2.8))
plt.plot(t6_adaptive, marker="o", label="adaptive", color="teal")
plt.plot(t6_decay, marker="s", label="decay", color="orange")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("coordinate")
plt.title("Toy 6 · AdamW components")
plt.legend()
plt.show()

▶ What you'll see: the adaptive step follows gradient sign, while decay follows parameter size.

### ✍️ Toy 7 · Normalization and optimizer state cost memory

Adaptive training also needs normalized signal scales and extra state arrays.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t7_signal = np.array([3.25, 2.25, 1.0, 0.0, -1.0, 4.0])  # -> [3.25, 2.25, 1.0, 0.0, -1.0, 4.0]

print("signals:", t7_signal.tolist())  # -> [3.25, 2.25, 1.0, 0.0, -1.0, 4.0]

t7_mean = 1.0  # -> 1.0

print("mean:", t7_mean)  # -> 1.0

t7_var = 0.25  # -> 0.25

print("variance:", t7_var)  # -> 0.25

t7_eps = 1e-5  # -> 1e-05

print("epsilon:", t7_eps)  # -> 1e-05

t7_normalized = (t7_signal - t7_mean) / np.sqrt(t7_var + t7_eps)  # -> [4.49991, 2.49995, 0.0, -1.99996, -3.99992, 5.99988]

print("normalized:", np.round(t7_normalized, 3).tolist())  # -> [4.5, 2.5, 0.0, -2.0, -4.0, 6.0]

t7_activation_vectors = 2  # -> 2

print("activation vectors:", t7_activation_vectors)  # -> 2

t7_width = 128  # -> 128

print("width:", t7_width)  # -> 128

t7_bytes = 4  # -> 4
t7_activation_kb = t7_activation_vectors * t7_width * t7_bytes / 1024  # -> 1.0

print("activation KB:", round(float(t7_activation_kb), 3))  # -> 1.0

t7_adam_state_kb = 2 * t7_width * t7_bytes / 1024  # -> 1.0

print("Adam state KB for one 128-vector:", round(float(t7_adam_state_kb), 3))  # -> 1.0

assert round(float(t7_normalized[0]), 3) == 4.5
assert round(float(t7_adam_state_kb), 3) == 1.0

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t7_signal.size), t7_normalized, color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("signal index")
plt.ylabel("standardized value")
plt.title("Toy 7 · normalized values")
plt.show()

▶ What you'll see: the largest signal is about `6` standard deviations high, and Adam state consumes extra memory.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for vectorized optimizer arithmetic and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for small plots that make optimizer behavior inspectable.
np.random.seed(0) # make all stochastic examples reproducible across notebook runs.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

## 🟢 Basics (warm-up)

### Basic 1 — Compute the lesson's one-step gradient descent update

**Goal.** Move one scalar parameter with vanilla gradient descent, because adaptive methods modify this same subtraction rule rather than replacing optimization entirely. We build it in 2 steps.

In [ ]:
theta_b1 = 2.0 # store the starting scalar parameter from the lesson arithmetic.
eta_b1 = 0.08 # store the learning rate used by the lesson.
g_b1 = 1.35 # store the local gradient used by the lesson.

print("theta:", theta_b1, "eta:", eta_b1, "gradient:", g_b1) # inspect the three ingredients before updating.

▶ What you'll see: the parameter, learning rate, and gradient are explicit before any arithmetic happens.

In [ ]:
step_b1 = eta_b1 * g_b1 # compute the amount subtracted from theta.
theta_next_b1 = theta_b1 - step_b1 # take one vanilla gradient descent step.

print("step:", round(step_b1, 3), "theta_next:", round(theta_next_b1, 3)) # inspect the update and result.

assert round(theta_next_b1, 3) == 1.892 # verify the lesson number 2.000 - 0.080*1.350.

▶ What you'll see: the step is `0.108`, so the parameter lands at `1.892`.

In [ ]:
theta_grid_b1 = np.linspace(1.75, 2.08, 120) # create a small one-dimensional loss slice around the update.
loss_grid_b1 = 0.5 * g_b1 * (theta_grid_b1 - theta_next_b1) ** 2 # sketch a local quadratic whose slope points from theta to theta_next.
plt.figure(figsize=(5, 3)) # create a compact before/after plot.
plt.plot(theta_grid_b1, loss_grid_b1, color="steelblue", label="local loss slice") # draw the loss surface.
plt.scatter([theta_b1, theta_next_b1], [0.5 * g_b1 * (theta_b1 - theta_next_b1) ** 2, 0.0], color=["crimson", "seagreen"], zorder=3) # mark start and updated points.
plt.annotate("start", (theta_b1, 0.5 * g_b1 * (theta_b1 - theta_next_b1) ** 2), xytext=(6, 8), textcoords="offset points") # label the original parameter.
plt.annotate("after step", (theta_next_b1, 0.0), xytext=(6, 8), textcoords="offset points") # label the updated parameter.
plt.title("Basic 1: one gradient descent step") # title the plot.
plt.xlabel("theta") # label the parameter axis.
plt.ylabel("local loss") # label the loss axis.
plt.legend() # show the curve label.
plt.show() # display the before/after movement.

▶ What you'll see: the red start point moves left to the green updated point on a small local loss slice.

👀 Takeaway: every adaptive optimizer still starts from the idea of subtracting a learning-rate-scaled gradient.

### Basic 2 — Compare two gradient coordinates

**Goal.** See why one global learning rate can be awkward, because different coordinates may have very different gradient scales. We build it in 2 steps.

In [ ]:
g_b2 = np.array([1.35, 0.05]) # define two gradient coordinates with very different magnitudes.
eta_b2 = 0.08 # use the same global learning rate for both coordinates.
vanilla_b2 = eta_b2 * g_b2 # compute vanilla coordinatewise update sizes.

print("gradients:", g_b2) # inspect raw coordinate scales.

▶ What you'll see: coordinate 0 is much steeper than coordinate 1.

In [ ]:
print("vanilla updates:", np.round(vanilla_b2, 4)) # inspect how raw gradients translate into movement.
print("update ratio:", round(float(vanilla_b2[0] / vanilla_b2[1]), 1)) # measure how uneven the movement is.

assert round(float(vanilla_b2[0] / vanilla_b2[1]), 1) == 27.0 # verify the scale contrast.
plt.figure(figsize=(4, 3)) # create a compact comparison chart.
plt.bar(["coord 0", "coord 1"], vanilla_b2, color="gray") # plot vanilla updates by coordinate.
plt.title("Basic 2: one eta, uneven steps") # title the chart.
plt.ylabel("update size") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: one coordinate receives a much larger update purely because its gradient is larger.

👀 Takeaway: adaptive optimizers are useful when gradient scale differs across parameters.

### Basic 3 — Normalize a coordinate by its current magnitude

**Goal.** Divide a gradient by a coordinatewise denominator, because this is the core shape of adaptive updates. We build it in 2 steps.

In [ ]:
g_b3 = np.array([1.35, 0.05]) # reuse the uneven two-coordinate gradient.
eps_b3 = 1e-8 # add a tiny value to avoid division by zero.
scale_b3 = np.sqrt(g_b3 ** 2) + eps_b3 # current-gradient magnitude used as a simple denominator.

print("scale:", np.round(scale_b3, 4)) # inspect one denominator per coordinate.

▶ What you'll see: each coordinate gets its own scale estimate.

In [ ]:
step_b3 = 0.08 * g_b3 / scale_b3 # compute a current-scale adaptive step.

print("scaled step:", np.round(step_b3, 3)) # inspect the normalized update.

assert np.allclose(np.round(step_b3, 3), np.array([0.08, 0.08])) # verify both positive coordinates move equally in this first-step toy.
plt.figure(figsize=(4, 3)) # create a compact chart.
plt.bar(["coord 0", "coord 1"], step_b3, color="seagreen") # plot scaled updates.
plt.title("Basic 3: divide by coordinate scale") # title the chart.
plt.ylabel("scaled update") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: the two positive coordinates now move by about the same amount.

👀 Takeaway: adaptive scaling changes the effective learning rate separately for every coordinate.

### Basic 4 — Accumulate squared gradients for AdaGrad

**Goal.** Build AdaGrad's memory variable, because it remembers how large each coordinate's gradients have been over all previous steps. We build it in 2 steps.

In [ ]:
grads_b4 = np.array([[1.2, 0.1], [1.0, 0.1], [0.8, 0.1]]) # create a short gradient history for two coordinates.
G_b4 = np.sum(grads_b4 ** 2, axis=0) # accumulate squared gradients coordinate by coordinate.

print("accumulated squares:", np.round(G_b4, 3)) # inspect AdaGrad memory after three steps.

▶ What you'll see: coordinate 0 has much larger accumulated history than coordinate 1.

In [ ]:
den_b4 = np.sqrt(G_b4) + 1e-8 # convert accumulated squared gradients into RMS-like denominators.

print("AdaGrad denominator:", np.round(den_b4, 3)) # inspect the scale used to divide future gradients.

assert np.allclose(np.round(G_b4, 2), np.array([3.08, 0.03])) # verify the hand-checkable accumulation.
plt.figure(figsize=(4, 3)) # create a memory chart.
plt.bar(["coord 0", "coord 1"], G_b4, color="purple") # plot accumulated squared-gradient memory.
plt.title("Basic 4: AdaGrad memory") # title the chart.
plt.ylabel("sum of squared gradients") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: the high-gradient coordinate has a much larger denominator waiting for it.

👀 Takeaway: AdaGrad shrinks frequently steep coordinates by letting their squared-gradient memory grow.

### Basic 5 — Take one AdaGrad step

**Goal.** Use AdaGrad's accumulated denominator in an update, because the history determines the effective step size. We build it in 3 steps.

In [ ]:
theta_b5 = np.array([2.0, 2.0]) # initialize two parameters.
g_b5 = np.array([1.0, 0.1]) # choose the current gradient.
G_prev_b5 = np.array([1.44, 0.01]) # use previous squared-gradient memory from earlier steps.

print("previous G:", G_prev_b5) # inspect memory before including the current gradient.

▶ What you'll see: coordinate 0 already has much more historical gradient energy.

In [ ]:
G_b5 = G_prev_b5 + g_b5 ** 2 # add the current squared gradient to AdaGrad memory.
step_b5 = 0.4 * g_b5 / (np.sqrt(G_b5) + 1e-8) # compute the AdaGrad update.

print("updated G:", np.round(G_b5, 3)) # inspect new memory.
print("AdaGrad step:", np.round(step_b5, 3)) # inspect coordinatewise movement.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
theta_next_b5 = theta_b5 - step_b5 # subtract the AdaGrad step from both parameters.

print("theta_next:", np.round(theta_next_b5, 3)) # inspect updated parameters.

assert round(float(step_b5[0]), 3) == 0.256 # verify coordinate 0's history-shrunk step.
plt.figure(figsize=(4, 3)) # create a compact update chart.
plt.bar(["coord 0", "coord 1"], step_b5, color="teal") # visualize AdaGrad updates.
plt.title("Basic 5: one AdaGrad update") # title the chart.
plt.ylabel("update size") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: the historically large-gradient coordinate is damped relative to the small-gradient coordinate.

👀 Takeaway: AdaGrad's denominator is cumulative, so past gradient scale directly controls today's movement.

### Basic 6 — Build RMSProp's exponential square average

**Goal.** Replace AdaGrad's permanent sum with a leaky average, because deep-learning gradient scales change over time. We build it in 2 steps.

In [ ]:
grads_b6 = np.array([1.2, 1.0, 0.2]) # create a one-coordinate gradient sequence that becomes smaller.
beta_b6 = 0.9 # choose the RMSProp memory coefficient.
v_b6 = 0.0 # initialize second-moment memory at zero.
history_b6 = [] # store v_t values.
for grad_b6 in grads_b6:
    v_b6 = beta_b6 * v_b6 + (1 - beta_b6) * grad_b6 ** 2 # update the leaky square average.
    history_b6.append(v_b6) # save the memory after this gradient.

print("v history:", np.round(history_b6, 4)) # inspect the exponential average.

▶ What you'll see: the memory rises after large gradients and then changes gradually.

In [ ]:
steps_b6 = 0.1 * grads_b6 / (np.sqrt(history_b6) + 1e-8) # compute RMSProp-style updates from the memory.

print("RMSProp steps:", np.round(steps_b6, 3)) # inspect how the denominator affects movement.

assert round(float(history_b6[0]), 3) == 0.144 # verify first EMA update 0.1*1.2^2.
plt.figure(figsize=(4, 3)) # create a compact line plot.
plt.plot(history_b6, marker="o", color="orange") # plot v_t over time.
plt.title("Basic 6: RMSProp square memory") # title the chart.
plt.xlabel("time step") # label the x-axis.
plt.ylabel("v_t") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: RMSProp's denominator is smoothed rather than permanently accumulated.

👀 Takeaway: RMSProp remembers recent gradient scale but slowly forgets stale history.

### Basic 7 — Compute Adam's first moment

**Goal.** Smooth the gradient direction with an exponential average, because Adam uses momentum to resist noisy minibatch reversals. We build it in 2 steps.

In [ ]:
grads_b7 = np.array([1.0, 0.6, -0.2]) # create gradients whose sign eventually changes.
beta1_b7 = 0.9 # choose Adam's standard first-moment decay.
m_b7 = 0.0 # initialize first moment at zero.
ms_b7 = [] # store raw first moments.
for grad_b7 in grads_b7:
    m_b7 = beta1_b7 * m_b7 + (1 - beta1_b7) * grad_b7 # update exponential average of gradients.
    ms_b7.append(m_b7) # save the raw first moment.

print("raw m values:", np.round(ms_b7, 4)) # inspect momentum memory.

▶ What you'll see: the first moment changes more slowly than the raw gradient sequence.

In [ ]:
mhat_b7 = np.array([ms_b7[t] / (1 - beta1_b7 ** (t + 1)) for t in range(len(ms_b7))]) # bias-correct raw moments.

print("bias-corrected m_hat:", np.round(mhat_b7, 4)) # inspect corrected direction estimates.

assert round(float(mhat_b7[0]), 3) == 1.0 # verify the first corrected value equals the first gradient.
plt.figure(figsize=(4, 3)) # create a direction plot.
plt.plot(grads_b7, marker="o", label="gradient") # plot raw gradients.
plt.plot(mhat_b7, marker="s", label="m_hat") # plot corrected first moment.
plt.title("Basic 7: Adam first moment") # title the chart.
plt.legend() # show line labels.
plt.show() # display the chart.

▶ What you'll see: `m_hat` smooths direction and does not instantly follow the negative gradient.

👀 Takeaway: Adam's first moment is a memory of direction, not just the current minibatch gradient.

### Basic 8 — Compute Adam's second moment

**Goal.** Track squared-gradient scale with an exponential average, because Adam divides by this scale like RMSProp. We build it in 2 steps.

In [ ]:
grads_b8 = np.array([1.0, 0.6, -0.2]) # reuse the same gradient sequence.
beta2_b8 = 0.999 # choose Adam's standard second-moment decay.
v_b8 = 0.0 # initialize second moment at zero.
vs_b8 = [] # store raw second moments.
for grad_b8 in grads_b8:
    v_b8 = beta2_b8 * v_b8 + (1 - beta2_b8) * grad_b8 ** 2 # update exponential average of squared gradients.
    vs_b8.append(v_b8) # save the raw second moment.

print("raw v values:", np.round(vs_b8, 6)) # inspect tiny raw values caused by beta2 near 1.

▶ What you'll see: raw `v` values start very small because the moving average is initialized at zero.

In [ ]:
vhat_b8 = np.array([vs_b8[t] / (1 - beta2_b8 ** (t + 1)) for t in range(len(vs_b8))]) # bias-correct the second moment.

print("bias-corrected v_hat:", np.round(vhat_b8, 4)) # inspect corrected square estimates.

assert round(float(vhat_b8[0]), 3) == 1.0 # verify first corrected second moment equals g^2.
plt.figure(figsize=(4, 3)) # create a scale-memory plot.
plt.plot(vhat_b8, marker="o", color="crimson") # plot corrected second moment values.
plt.title("Basic 8: Adam second moment") # title the chart.
plt.xlabel("time step") # label the x-axis.
plt.ylabel("v_hat") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: corrected squared-gradient scale remains positive even when the gradient sign changes.

👀 Takeaway: Adam's second moment measures magnitude/volatility, not direction.

### Basic 9 — Take one Adam step

**Goal.** Combine corrected first and second moments into an Adam update, because this is the optimizer's central formula. We build it in 3 steps.

In [ ]:
g_b9 = 1.35 # choose the lesson gradient for the first Adam step.
b1_b9, b2_b9 = 0.9, 0.999 # use standard Adam decay rates.
m_b9 = (1 - b1_b9) * g_b9 # first raw moment at t=1.
v_b9 = (1 - b2_b9) * g_b9 ** 2 # second raw moment at t=1.

print("raw m:", round(m_b9, 4), "raw v:", round(v_b9, 6)) # inspect zero-biased memories.

▶ What you'll see: raw memories are much smaller than the gradient and squared gradient.

In [ ]:
mhat_b9 = m_b9 / (1 - b1_b9) # bias-correct first moment at t=1.
vhat_b9 = v_b9 / (1 - b2_b9) # bias-correct second moment at t=1.
step_b9 = 0.08 * mhat_b9 / (np.sqrt(vhat_b9) + 1e-8) # compute Adam's adaptive step.

print("m_hat:", round(mhat_b9, 3), "v_hat:", round(vhat_b9, 3), "step:", round(step_b9, 3)) # inspect the corrected update.

assert round(step_b9, 3) == 0.08 # verify first-step Adam scale for a positive scalar gradient.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
theta_b9 = 2.0 # initialize a scalar parameter.
theta_next_b9 = theta_b9 - step_b9 # apply the Adam step.

print("theta_next:", round(theta_next_b9, 3)) # inspect updated parameter.

plt.figure(figsize=(4, 3)) # create a small chart.
plt.bar(["GD step", "Adam step"], [0.08 * g_b9, step_b9], color=["gray", "teal"]) # compare vanilla and Adam first-step sizes.
plt.title("Basic 9: Adam first step") # title the chart.
plt.ylabel("update size") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: Adam's first step is learning-rate sized (`0.08`) while vanilla GD would step `0.108`.

👀 Takeaway: Adam uses momentum for direction and RMS scale for coordinatewise normalization.

### Basic 10 — Add epsilon for numerical stability

**Goal.** Protect division by a tiny denominator, because adaptive optimizers must behave when a coordinate has nearly zero squared-gradient history. We build it in 2 steps.

In [ ]:
g_b10 = np.array([0.0, 1e-12]) # create nearly zero gradients.
vhat_b10 = g_b10 ** 2 # second-moment estimate is exactly or nearly zero.
eps_b10 = 1e-8 # numerical stabilizer added to the denominator.

print("sqrt(vhat):", np.sqrt(vhat_b10)) # inspect the unsafe denominator before epsilon.

▶ What you'll see: at least one coordinate has a zero denominator without the stabilizer.

In [ ]:
safe_step_b10 = 0.08 * g_b10 / (np.sqrt(vhat_b10) + eps_b10) # compute a safe adaptive step.

print("safe step:", safe_step_b10) # inspect finite update values.

assert np.all(np.isfinite(safe_step_b10)) # verify no NaN or inf appears.
plt.figure(figsize=(4, 3)) # create a compact stability chart.
plt.bar(["zero grad", "tiny grad"], safe_step_b10, color="orange") # plot stable updates.
plt.title("Basic 10: epsilon keeps division finite") # title the chart.
plt.ylabel("safe update") # label the y-axis.
plt.show() # display the chart.

▶ What you'll see: epsilon prevents undefined division and keeps updates finite.

👀 Takeaway: epsilon is small mathematically but crucial for robust optimizer code.

## 🟡 Easy

### Easy 1 — Compare SGD, AdaGrad, RMSProp, and Adam on one gradient sequence

**Goal.** Run four optimizers on the same scalar gradients, because comparing trajectories shows what the adaptive denominator changes. We build it in 4 steps.

In [ ]:
grads_e1 = np.array([1.35, 1.0, 0.5, -0.2, -0.1]) # define one scalar gradient history.
eta_e1 = 0.08 # use the lesson learning-rate scale for every optimizer.

print("gradient sequence:", grads_e1) # inspect the shared input to all methods.

▶ What you'll see: gradients start positive and later turn slightly negative.

In [ ]:
t_sgd_e1 = 2.0 # initialize SGD parameter.
t_ag_e1 = 2.0 # initialize AdaGrad parameter.
t_rms_e1 = 2.0 # initialize RMSProp parameter.
t_adam_e1 = 2.0 # initialize Adam parameter.
G_e1 = 0.0; v_rms_e1 = 0.0; m_e1 = 0.0; v_adam_e1 = 0.0 # initialize optimizer states.
traj_e1 = [[t_sgd_e1, t_ag_e1, t_rms_e1, t_adam_e1]] # store trajectories including the start.

print("initial theta:", traj_e1[0]) # inspect starting point.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
for t_e1, g_e1 in enumerate(grads_e1, start=1): # process the same gradient with every optimizer.
    t_sgd_e1 -= eta_e1 * g_e1 # vanilla SGD step.
    G_e1 += g_e1 ** 2 # AdaGrad squared-gradient memory.
    t_ag_e1 -= eta_e1 * g_e1 / (np.sqrt(G_e1) + 1e-8) # AdaGrad step.
    v_rms_e1 = 0.9 * v_rms_e1 + 0.1 * g_e1 ** 2 # RMSProp square memory.
    t_rms_e1 -= eta_e1 * g_e1 / (np.sqrt(v_rms_e1) + 1e-8) # RMSProp step.
    m_e1 = 0.9 * m_e1 + 0.1 * g_e1 # Adam first moment.
    v_adam_e1 = 0.999 * v_adam_e1 + 0.001 * g_e1 ** 2 # Adam second moment.
    mh_e1 = m_e1 / (1 - 0.9 ** t_e1) # Adam bias-corrected first moment.
    vh_e1 = v_adam_e1 / (1 - 0.999 ** t_e1) # Adam bias-corrected second moment.
    t_adam_e1 -= eta_e1 * mh_e1 / (np.sqrt(vh_e1) + 1e-8) # Adam step.
    traj_e1.append([t_sgd_e1, t_ag_e1, t_rms_e1, t_adam_e1]) # save all parameters.
traj_e1 = np.array(traj_e1) # convert to an array for plotting.

print("final thetas:", np.round(traj_e1[-1], 3)) # inspect where each optimizer ends.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a trajectory comparison.
for j_e1, name_e1 in enumerate(["SGD", "AdaGrad", "RMSProp", "Adam"]): # plot each optimizer.
    plt.plot(traj_e1[:, j_e1], marker="o", label=name_e1) # draw one parameter trajectory.
plt.title("Easy 1: optimizer trajectories on one gradient sequence") # title the figure.
plt.xlabel("step") # label the x-axis.
plt.ylabel("theta") # label the y-axis.
plt.legend() # show method names.
plt.show() # display the comparison.

▶ What you'll see: all methods move downhill, but adaptive methods take different-sized steps from the same gradients.

👀 Takeaway: optimizer choice changes the parameter path even when the gradient sequence is identical.

### Easy 2 — Optimize an anisotropic quadratic

**Goal.** Minimize a two-coordinate bowl with very different curvature, because adaptive scaling helps when one direction is much steeper than another. We build it in 4 steps.

In [ ]:
theta_e2 = np.array([4.0, 4.0]) # start far from the minimum at zero.
curv_e2 = np.array([20.0, 1.0]) # coordinate 0 is twenty times steeper than coordinate 1.

print("start theta:", theta_e2, "curvatures:", curv_e2) # inspect the quadratic geometry.

▶ What you'll see: both coordinates start equal, but the loss is much steeper in coordinate 0.

In [ ]:
t_sgd_e2 = theta_e2.copy() # initialize SGD state.
t_adam_e2 = theta_e2.copy() # initialize Adam-like state.
m_e2 = np.zeros(2); v_e2 = np.zeros(2) # initialize Adam moments.
path_sgd_e2 = [t_sgd_e2.copy()]; path_adam_e2 = [t_adam_e2.copy()] # store paths.
for step_e2 in range(40): # run a short optimization.
    g_sgd_e2 = curv_e2 * t_sgd_e2 # gradient of 0.5*sum(curv*theta^2).
    t_sgd_e2 -= 0.02 * g_sgd_e2 # SGD needs a small rate due to steep coordinate.
    g_adam_e2 = curv_e2 * t_adam_e2 # same gradient for Adam-like method.
    m_e2 = 0.9 * m_e2 + 0.1 * g_adam_e2 # first moment.
    v_e2 = 0.999 * v_e2 + 0.001 * g_adam_e2 ** 2 # second moment.
    mh_e2 = m_e2 / (1 - 0.9 ** (step_e2 + 1)) # bias correction.
    vh_e2 = v_e2 / (1 - 0.999 ** (step_e2 + 1)) # bias correction.
    t_adam_e2 -= 0.12 * mh_e2 / (np.sqrt(vh_e2) + 1e-8) # coordinatewise adaptive step.
    path_sgd_e2.append(t_sgd_e2.copy()); path_adam_e2.append(t_adam_e2.copy()) # store paths.
path_sgd_e2 = np.array(path_sgd_e2); path_adam_e2 = np.array(path_adam_e2) # convert for plotting.

print("final SGD:", np.round(path_sgd_e2[-1], 3), "final Adam-like:", np.round(path_adam_e2[-1], 3)) # inspect endpoints.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
loss_sgd_e2 = 0.5 * np.sum(curv_e2 * path_sgd_e2 ** 2, axis=1) # compute SGD losses along path.
loss_adam_e2 = 0.5 * np.sum(curv_e2 * path_adam_e2 ** 2, axis=1) # compute Adam-like losses along path.

print("final losses:", round(float(loss_sgd_e2[-1]), 4), round(float(loss_adam_e2[-1]), 4)) # inspect convergence.

assert loss_adam_e2[-1] < loss_sgd_e2[-1] # verify the adaptive method is better in this toy setup.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a loss curve.
plt.plot(loss_sgd_e2, label="SGD") # plot SGD loss.
plt.plot(loss_adam_e2, label="Adam-like") # plot adaptive loss.
plt.yscale("log") # use log scale to show both curves.
plt.title("Easy 2: adaptive scaling on an anisotropic bowl") # title the figure.
plt.xlabel("step") # label x-axis.
plt.ylabel("loss, log scale") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: adaptive scaling progresses on both coordinates while SGD is constrained by the steep direction.

👀 Takeaway: per-coordinate denominators are especially useful when curvature or gradient scale is uneven.

### Easy 3 — Show bias correction in Adam

**Goal.** Compare raw and bias-corrected moments, because Adam's zero initialization otherwise underestimates early gradient statistics. We build it in 3 steps.

In [ ]:
grads_e3 = np.array([1.35, 1.0, 0.5]) # define early gradients where zero-start bias matters most.
b1_e3, b2_e3 = 0.9, 0.999 # standard Adam betas.
m_e3 = 0.0; v_e3 = 0.0 # initialize moment estimates at zero.
raw_e3 = []; corr_e3 = [] # store raw and corrected values.

print("gradients:", grads_e3) # inspect inputs.

▶ What you'll see: the first few gradients are positive and fairly large.

In [ ]:
for t_e3, grad_e3 in enumerate(grads_e3, start=1): # update Adam moments through early steps.
    m_e3 = b1_e3 * m_e3 + (1 - b1_e3) * grad_e3 # raw first moment.
    v_e3 = b2_e3 * v_e3 + (1 - b2_e3) * grad_e3 ** 2 # raw second moment.
    raw_e3.append([m_e3, v_e3]) # store raw moments.
    corr_e3.append([m_e3 / (1 - b1_e3 ** t_e3), v_e3 / (1 - b2_e3 ** t_e3)]) # store corrected moments.
raw_e3 = np.array(raw_e3); corr_e3 = np.array(corr_e3) # convert to arrays.

print("raw moments:\n", np.round(raw_e3, 5)) # inspect biased low estimates.
print("corrected moments:\n", np.round(corr_e3, 5)) # inspect corrected estimates.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert round(float(corr_e3[0, 0]), 3) == 1.35 # verify first corrected first moment.
assert round(float(corr_e3[0, 1]), 4) == round(1.35 ** 2, 4) # verify first corrected second moment.
plt.figure(figsize=(5, 3)) # create a comparison chart.
plt.plot(raw_e3[:, 0], marker="o", label="raw m") # plot raw m.
plt.plot(corr_e3[:, 0], marker="s", label="corrected m_hat") # plot corrected m.
plt.title("Easy 3: Adam bias correction") # title the chart.
plt.xlabel("early step") # label x-axis.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: raw `m` is much smaller than `m_hat` early because it started from zero.

👀 Takeaway: bias correction makes Adam's early steps reflect the observed gradients rather than the zero initial state.

### Easy 4 — Compare Adam and AdamW weight decay

**Goal.** Separate adaptive gradient scaling from parameter shrinkage, because AdamW decouples regularization from gradient history. We build it in 3 steps.

In [ ]:
theta_e4 = np.array([2.0, -0.5]) # define two parameters.
g_e4 = np.array([1.35, 0.05]) # define current gradients.
eta_e4 = 0.08; wd_e4 = 0.1 # choose learning rate and weight decay.

print("theta:", theta_e4, "gradient:", g_e4) # inspect inputs.

▶ What you'll see: the two coordinates have different parameter sizes and gradient magnitudes.

In [ ]:
# Coupled L2-style Adam first step: add wd*theta to the gradient before adaptive scaling.
g_coupled_e4 = g_e4 + wd_e4 * theta_e4 # form a coupled gradient.
step_coupled_e4 = eta_e4 * g_coupled_e4 / (np.sqrt(g_coupled_e4 ** 2) + 1e-8) # first-step adaptive scaling.
theta_coupled_e4 = theta_e4 - step_coupled_e4 # update with coupled regularization.
# AdamW: adapt the data gradient, then subtract a separate decay term.
step_data_e4 = eta_e4 * g_e4 / (np.sqrt(g_e4 ** 2) + 1e-8) # adaptive data-gradient step.
theta_adamw_e4 = theta_e4 - step_data_e4 - eta_e4 * wd_e4 * theta_e4 # decoupled update.

print("coupled next:", np.round(theta_coupled_e4, 3)) # inspect coupled result.
print("AdamW next:", np.round(theta_adamw_e4, 3)) # inspect decoupled result.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert np.allclose(np.round(theta_adamw_e4, 3), np.array([1.904, -0.576])) # verify AdamW arithmetic.
plt.figure(figsize=(5, 3)) # create a comparison chart.
plt.bar(["coupled coord0", "AdamW coord0"], [theta_coupled_e4[0], theta_adamw_e4[0]], color=["gray", "teal"]) # compare coordinate 0 after update.
plt.title("Easy 4: coupled decay vs AdamW") # title the chart.
plt.ylabel("updated parameter") # label y-axis.
plt.show() # display the chart.

▶ What you'll see: coupled and decoupled updates can differ even with the same learning rate and decay coefficient.

👀 Takeaway: AdamW makes weight decay a direct parameter shrinkage term rather than an adaptive-gradient side effect.

### Easy 5 — Normalize a signal and compute activation memory

**Goal.** Reproduce the lesson's scale and memory bookkeeping, because optimizer stability is tied to numerical scale and practical hardware cost. We build it in 3 steps.

In [ ]:
signal_e5 = 3.25 # use the lesson affine/gated signal.
mean_e5 = 1.0 # normalization mean.
var_e5 = 0.25 # normalization variance.
eps_e5 = 1e-5 # normalization stabilizer.

print("signal:", signal_e5, "mean:", mean_e5, "variance:", var_e5) # inspect normalization inputs.

▶ What you'll see: the signal is above the reference mean.

In [ ]:
normed_e5 = (signal_e5 - mean_e5) / np.sqrt(var_e5 + eps_e5) # compute normalized value.

print("normalized value:", round(float(normed_e5), 3)) # inspect scale after normalization.

assert round(float(normed_e5), 3) == 4.5 # verify the lesson number.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
vectors_e5 = 2; length_e5 = 128; bytes_per_float_e5 = 4 # define tiny activation-block dimensions.
mem_kb_e5 = vectors_e5 * length_e5 * bytes_per_float_e5 / 1024 # compute float32 memory in KB.

print("memory KB:", round(mem_kb_e5, 3)) # inspect hardware footprint.

assert round(mem_kb_e5, 3) == 1.0 # verify the lesson memory number.
plt.figure(figsize=(4, 3)) # create a two-metric chart.
plt.bar(["normalized", "memory KB"], [normed_e5, mem_kb_e5], color=["purple", "orange"]) # compare the scale and memory values.
plt.title("Easy 5: scale and memory bookkeeping") # title the chart.
plt.show() # display the chart.

▶ What you'll see: the normalized signal is large (`4.5`), while the toy activation block uses `1 KB`.

👀 Takeaway: adaptive optimization lives inside a larger system where numerical scale and memory both matter.

## 🔴 Advanced

### Advanced 1 — Train linear regression with multiple optimizers

**Goal.** Fit the same tiny linear model with SGD, RMSProp, and Adam, because end-to-end training shows how update rules affect loss curves. We build it in 5 steps.

In [ ]:
x_a1 = np.linspace(-1, 1, 40) # create one-dimensional inputs.
y_a1 = 2.0 * x_a1 - 0.5 + 0.1 * np.sin(5 * x_a1) # create deterministic targets with slight nonlinearity.

print("x range:", round(float(x_a1.min()), 1), "to", round(float(x_a1.max()), 1)) # inspect data range.

▶ What you'll see: the training data spans inputs from -1 to 1.

In [ ]:
def loss_and_grad_a1(params_a1): # compute MSE loss and gradient for yhat = w*x + b.
    w_a1, b_a1 = params_a1 # unpack parameters.
    pred_a1 = w_a1 * x_a1 + b_a1 # compute predictions.
    err_a1 = pred_a1 - y_a1 # residuals.
    loss_a1 = float(np.mean(err_a1 ** 2)) # mean squared error.
    grad_a1 = np.array([2 * np.mean(err_a1 * x_a1), 2 * np.mean(err_a1)]) # gradients for w and b.
    return loss_a1, grad_a1 # return both objective and gradient.

print("initial loss at zeros:", round(loss_and_grad_a1(np.array([0.0, 0.0]))[0], 3)) # inspect starting loss.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
params_sgd_a1 = np.array([0.0, 0.0]); params_rms_a1 = np.array([0.0, 0.0]); params_adam_a1 = np.array([0.0, 0.0]) # initialize all methods equally.
v_rms_a1 = np.zeros(2); m_adam_a1 = np.zeros(2); v_adam_a1 = np.zeros(2) # initialize optimizer states.
curves_a1 = {"SGD": [], "RMSProp": [], "Adam": []} # store loss curves.
for t_a1 in range(1, 121): # run full-batch optimization.
    loss_sgd_a1, grad_sgd_a1 = loss_and_grad_a1(params_sgd_a1) # compute SGD gradient.
    params_sgd_a1 -= 0.25 * grad_sgd_a1 # update SGD.
    loss_rms_a1, grad_rms_a1 = loss_and_grad_a1(params_rms_a1) # compute RMSProp gradient.
    v_rms_a1 = 0.9 * v_rms_a1 + 0.1 * grad_rms_a1 ** 2 # update RMSProp second moment.
    params_rms_a1 -= 0.05 * grad_rms_a1 / (np.sqrt(v_rms_a1) + 1e-8) # update RMSProp.
    loss_adam_a1, grad_adam_a1 = loss_and_grad_a1(params_adam_a1) # compute Adam gradient.
    m_adam_a1 = 0.9 * m_adam_a1 + 0.1 * grad_adam_a1 # Adam first moment.
    v_adam_a1 = 0.999 * v_adam_a1 + 0.001 * grad_adam_a1 ** 2 # Adam second moment.
    mh_a1 = m_adam_a1 / (1 - 0.9 ** t_a1); vh_a1 = v_adam_a1 / (1 - 0.999 ** t_a1) # bias correction.
    params_adam_a1 -= 0.08 * mh_a1 / (np.sqrt(vh_a1) + 1e-8) # update Adam.
    curves_a1["SGD"].append(loss_sgd_a1); curves_a1["RMSProp"].append(loss_rms_a1); curves_a1["Adam"].append(loss_adam_a1) # record losses.

print("final losses:", {k_a1: round(v_a1[-1], 4) for k_a1, v_a1 in curves_a1.items()}) # inspect final losses.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert curves_a1["Adam"][-1] < curves_a1["Adam"][0] # verify Adam reduced training loss.
assert curves_a1["RMSProp"][-1] < curves_a1["RMSProp"][0] # verify RMSProp reduced training loss.

print("final Adam params:", np.round(params_adam_a1, 3)) # inspect learned slope and intercept.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a training-curve plot.
for name_a1, curve_a1 in curves_a1.items(): # plot each optimizer curve.
    plt.plot(curve_a1, label=name_a1) # draw loss over iterations.
plt.yscale("log") # log scale shows early and late progress.
plt.title("Advanced 1: optimizer loss curves") # title the figure.
plt.xlabel("step") # label x-axis.
plt.ylabel("MSE, log scale") # label y-axis.
plt.legend() # show method names.
plt.show() # display the chart.

▶ What you'll see: all methods reduce loss, but their curves descend with different speeds and smoothness.

👀 Takeaway: adaptive optimizers are easiest to understand by watching both the formula and the resulting training curve.

### Advanced 2 — Inspect effective learning rates per coordinate

**Goal.** Track `eta / sqrt(v_hat)` in Adam, because adaptive methods can be read as changing each coordinate's effective learning rate over time. We build it in 4 steps.

In [ ]:
grads_a2 = np.column_stack([np.linspace(2.0, 0.2, 30), np.full(30, 0.1)]) # create large-decaying and small-steady gradients.
eta_a2 = 0.08 # base learning rate.

print("gradient matrix shape:", grads_a2.shape) # inspect time by coordinate layout.

▶ What you'll see: there are 30 time steps and 2 parameter coordinates.

In [ ]:
m_a2 = np.zeros(2); v_a2 = np.zeros(2) # initialize Adam moments.
eff_lr_a2 = [] # store effective learning rates.
for t_a2, grad_a2 in enumerate(grads_a2, start=1): # process gradients over time.
    m_a2 = 0.9 * m_a2 + 0.1 * grad_a2 # update first moment for completeness.
    v_a2 = 0.999 * v_a2 + 0.001 * grad_a2 ** 2 # update second moment.
    vh_a2 = v_a2 / (1 - 0.999 ** t_a2) # bias-correct scale estimate.
    eff_lr_a2.append(eta_a2 / (np.sqrt(vh_a2) + 1e-8)) # effective multiplier before m_hat.
eff_lr_a2 = np.array(eff_lr_a2) # convert to array.

print("first effective lr:", np.round(eff_lr_a2[0], 3)) # inspect starting multipliers.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
print("last effective lr:", np.round(eff_lr_a2[-1], 3)) # inspect ending multipliers.

assert eff_lr_a2[-1, 1] > eff_lr_a2[-1, 0] # verify the small-gradient coordinate gets the larger multiplier.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create an effective-learning-rate plot.
plt.plot(eff_lr_a2[:, 0], label="coord 0: large gradients") # plot steep coordinate multiplier.
plt.plot(eff_lr_a2[:, 1], label="coord 1: small gradients") # plot small coordinate multiplier.
plt.title("Advanced 2: Adam effective learning rates") # title the figure.
plt.xlabel("step") # label x-axis.
plt.ylabel("eta / sqrt(v_hat)") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the small-gradient coordinate receives a larger multiplier because its second moment is smaller.

👀 Takeaway: adaptive optimizers do not use one fixed learning rate; they create a time-varying diagonal learning-rate matrix.

### Advanced 3 — Compare noisy minibatch gradients with momentum smoothing

**Goal.** Smooth noisy gradient estimates, because minibatches estimate a full-data gradient and can bounce from step to step. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(0) # create reproducible noise.
true_grad_a3 = 0.4 # set a positive full-data gradient.
noise_a3 = rng_a3.normal(0, 0.35, size=60) # simulate minibatch noise.
grad_seq_a3 = true_grad_a3 + noise_a3 # noisy minibatch gradients.

print("mean noisy gradient:", round(float(np.mean(grad_seq_a3)), 3)) # inspect that noise averages near the true gradient.

▶ What you'll see: individual gradients are noisy, but the average is close to the true direction.

In [ ]:
m_a3 = 0.0 # initialize momentum memory.
smoothed_a3 = [] # store exponential averages.
for grad_a3 in grad_seq_a3:
    m_a3 = 0.9 * m_a3 + 0.1 * grad_a3 # update first moment.
    smoothed_a3.append(m_a3) # save the smoothed value.
smoothed_a3 = np.array(smoothed_a3) # convert to array.

print("raw std:", round(float(np.std(grad_seq_a3)), 3), "smoothed std:", round(float(np.std(smoothed_a3[10:])), 3)) # compare variability after warmup.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert np.std(smoothed_a3[10:]) < np.std(grad_seq_a3) # verify smoothing reduces variance after warmup.
steps_raw_a3 = 0.08 * grad_seq_a3 # raw SGD update sizes.
steps_smooth_a3 = 0.08 * smoothed_a3 # momentum-smoothed update sizes.

print("first five raw steps:", np.round(steps_raw_a3[:5], 3)) # inspect noisy updates.
print("first five smoothed steps:", np.round(steps_smooth_a3[:5], 3)) # inspect smoothed updates.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a noisy-gradient plot.
plt.plot(grad_seq_a3, alpha=0.45, label="noisy minibatch gradient") # plot raw gradients.
plt.plot(smoothed_a3, linewidth=2, label="momentum first moment") # plot smoothed gradients.
plt.axhline(true_grad_a3, color="black", linestyle="--", label="true gradient") # show full-data reference.
plt.title("Advanced 3: momentum smooths minibatch noise") # title the figure.
plt.xlabel("minibatch step") # label x-axis.
plt.ylabel("gradient estimate") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the momentum curve wiggles less than the raw minibatch gradients and stays near the true direction.

👀 Takeaway: Adam's first moment helps turn noisy local gradient estimates into steadier global training movement.

### Advanced 4 — Demonstrate AdaGrad's late-training stall

**Goal.** Show how permanent squared-gradient accumulation can make later steps tiny, because AdaGrad never forgets early large gradients. We build it in 4 steps.

In [ ]:
grads_a4 = np.r_[np.full(10, 2.0), np.full(30, 0.2)] # create large early gradients followed by small persistent gradients.
eta_a4 = 0.5 # choose a visible base learning rate.

print("gradient phases:", grads_a4[:3], "...", grads_a4[-3:]) # inspect early and late values.

▶ What you'll see: the sequence starts with large gradients, then settles into small gradients.

In [ ]:
G_a4 = 0.0; v_a4 = 0.0 # initialize AdaGrad and RMSProp square memories.
steps_ag_a4 = []; steps_rms_a4 = [] # store step magnitudes.
for grad_a4 in grads_a4:
    G_a4 += grad_a4 ** 2 # AdaGrad permanent accumulation.
    steps_ag_a4.append(eta_a4 * grad_a4 / (np.sqrt(G_a4) + 1e-8)) # AdaGrad step.
    v_a4 = 0.9 * v_a4 + 0.1 * grad_a4 ** 2 # RMSProp leaky memory.
    steps_rms_a4.append(eta_a4 * grad_a4 / (np.sqrt(v_a4) + 1e-8)) # RMSProp step.
steps_ag_a4 = np.array(steps_ag_a4); steps_rms_a4 = np.array(steps_rms_a4) # convert to arrays.

print("late AdaGrad step:", round(float(steps_ag_a4[-1]), 4), "late RMSProp step:", round(float(steps_rms_a4[-1]), 4)) # inspect late movement.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert steps_ag_a4[-1] < steps_rms_a4[-1] # verify permanent memory gives the smaller late step.

print("final AdaGrad memory:", round(G_a4, 3), "final RMSProp memory:", round(v_a4, 3)) # inspect memory sizes.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a late-training comparison.
plt.plot(steps_ag_a4, label="AdaGrad") # plot AdaGrad steps.
plt.plot(steps_rms_a4, label="RMSProp") # plot RMSProp steps.
plt.title("Advanced 4: permanent memory can stall AdaGrad") # title the figure.
plt.xlabel("step") # label x-axis.
plt.ylabel("update magnitude") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: AdaGrad's late steps are much smaller because early large gradients remain in the denominator forever.

👀 Takeaway: AdaGrad is powerful for sparse features, but RMSProp/Adam are often preferred in deep nets because they forget stale scale.

### Advanced 5 — Stress-test AdamW on scale and decay

**Goal.** Sweep weight decay values for AdamW on a tiny regression task, because regularization is a separate capacity knob from adaptive gradient scaling. We build it in 5 steps.

In [ ]:
x_a5 = np.linspace(-1, 1, 30) # create deterministic inputs.
y_a5 = 1.5 * x_a5 + 0.2 # create linear targets.
wd_grid_a5 = np.array([0.0, 0.01, 0.1, 0.5]) # choose decoupled weight-decay strengths.

print("weight decay grid:", wd_grid_a5) # inspect sweep values.

▶ What you'll see: the experiment will train one AdamW model for each decay value.

In [ ]:
def grad_loss_a5(params_a5): # compute MSE loss and gradients for linear regression.
    pred_a5 = params_a5[0] * x_a5 + params_a5[1] # predictions.
    err_a5 = pred_a5 - y_a5 # residuals.
    loss_a5 = float(np.mean(err_a5 ** 2)) # MSE.
    grad_a5 = np.array([2 * np.mean(err_a5 * x_a5), 2 * np.mean(err_a5)]) # gradient for slope and bias.
    return loss_a5, grad_a5 # return loss and gradient.

print("zero-param loss:", round(grad_loss_a5(np.zeros(2))[0], 3)) # inspect starting loss.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
final_losses_a5 = []; final_norms_a5 = [] # store results for each decay.
for wd_a5 in wd_grid_a5: # train one model per decay value.
    params_a5 = np.array([0.0, 0.0]) # reset parameters.
    m_a5 = np.zeros(2); v_a5 = np.zeros(2) # reset Adam states.
    for t_a5 in range(1, 121): # run AdamW updates.
        loss_a5, grad_a5 = grad_loss_a5(params_a5) # compute current gradient.
        m_a5 = 0.9 * m_a5 + 0.1 * grad_a5 # first moment.
        v_a5 = 0.999 * v_a5 + 0.001 * grad_a5 ** 2 # second moment.
        mh_a5 = m_a5 / (1 - 0.9 ** t_a5); vh_a5 = v_a5 / (1 - 0.999 ** t_a5) # bias correction.
        params_a5 -= 0.05 * mh_a5 / (np.sqrt(vh_a5) + 1e-8) # adaptive data-gradient step.
        params_a5 -= 0.05 * wd_a5 * params_a5 # decoupled weight decay step.
    final_losses_a5.append(grad_loss_a5(params_a5)[0]) # store final loss.
    final_norms_a5.append(float(np.linalg.norm(params_a5))) # store parameter norm.

print("final losses:", np.round(final_losses_a5, 4)) # inspect fit quality.
print("final norms:", np.round(final_norms_a5, 3)) # inspect capacity shrinkage.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
assert final_norms_a5[-1] < final_norms_a5[0] # verify stronger decay shrinks parameters more.

print("best decay by loss:", float(wd_grid_a5[int(np.argmin(final_losses_a5))])) # inspect best value in this tiny sweep.

▶ What you'll see: inspect the printed values and any plot to connect the arithmetic with the optimizer behavior before moving on.

In [ ]:
plt.figure(figsize=(5, 3)) # create a sweep chart.
plt.plot(wd_grid_a5, final_losses_a5, marker="o", label="final MSE") # plot loss by decay.
plt.plot(wd_grid_a5, np.array(final_norms_a5) / max(final_norms_a5), marker="s", label="scaled ||theta||") # plot scaled norm.
plt.title("Advanced 5: AdamW decay sweep") # title the figure.
plt.xlabel("weight decay") # label x-axis.
plt.ylabel("value") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: stronger weight decay shrinks parameters, and too much decay can harm the fit.

👀 Takeaway: AdamW lets you tune adaptive optimization and capacity control as related but distinct mechanisms.